In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
df = pd.read_csv('diabetes.csv')

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Pregnancies    768 non-null    int64  
 1   Glucose        768 non-null    int64  
 2   BloodPressure  768 non-null    int64  
 3   SkinThickness  768 non-null    int64  
 4   Insulin        768 non-null    int64  
 5   BMI            768 non-null    float64
 6   Pedigree       768 non-null    float64
 7   Age            768 non-null    int64  
 8   Outcome        768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [22]:
X_before = df.drop('Outcome', axis=1)
y = df['Outcome']

(X - X min) / (X max -  X min)
(X - X mean) / X std

In [23]:
X = (X_before - X_before.min()) / (X_before.max() - X_before.min())

In [24]:
np.random.seed(42)

idx_0 = np.where(y == 0)[0]
idx_1 = np.where(y == 1)[0]

np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

train_0 = int(len(idx_0) * 0.8)
train_1 = int(len(idx_1) * 0.8)

train_idx = np.concatenate([idx_0[:train_0], idx_1[:train_1]])
test_idx = np.concatenate([idx_0[train_0:], idx_1[train_1:]])

np.random.shuffle(train_idx)
np.random.shuffle(test_idx)

X_train = X.iloc[train_idx].values
y_train = y.iloc[train_idx].values
X_test = X.iloc[test_idx].values
y_test = y.iloc[test_idx].values

In [25]:
def knn(X_train, y_train, X_test, k=3, P=2, weight='uniform'):
    diff = np.abs(X_train - X_test)
    distance = np.sum(diff ** P, axis=1)** (1/P)
    
    sort = np.argsort(distance)
    nearest = sort[:k]
    nearest_label = y_train[nearest]
    nearest_distance = distance[nearest]

    if weight == 'uniform':
        label, count = np.unique(nearest_label.astype(int), return_counts=True)
        prediction = label[count.argmax()]
    elif weight == 'weighted':
        epsilon = 1e+5
        bobot = 1 / (nearest_distance + epsilon)
        total_bobot = np.bincount(nearest_label.astype(int), weights=bobot)
        prediction = total_bobot.argmax()

    return prediction

In [32]:
vote = ['uniform', 'weighted']
result = {}

for method in vote:
    correct = 0
    total = len(X_test)

    for i in range(total):
        pred = knn(X_train, y_train, X_test[i], k=3, weight=method)
        if pred == y_test[i]:
            correct += 1
    acc = correct / total
    result[method] = acc
    print(f'Method: {method} - Akurasi: {acc:.4f}')

Method: uniform - Akurasi: 0.7208
Method: weighted - Akurasi: 0.7208
